In [5]:
import sys

sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from utils import cross_validation_split, fix_seed, gerar_dados_sinteticos_queimadas

fix_seed(42)

# Se você já tiver o CSV do GOV, substitua por: pd.read_csv('../data/queimadas_meteorologia.csv')
df = gerar_dados_sinteticos_queimadas(n_samples=1000)

features = [
    "dias_sem_chuva",
    "precipitacao",
    "temperatura_max",
    "umidade_relativa",
    "velocidade_vento",
]
X = df[features].copy()
y = df["risco_critico"].copy()

print(f"Dataset carregado: {X.shape[0]} amostras, {X.shape[1]} atributos.")
print(f"Distribuição de classes (risco_critico):\n{y.value_counts(normalize=True)}")

Dataset carregado: 1000 amostras, 5 atributos.
Distribuição de classes (risco_critico):
risco_critico
0    0.998
1    0.002
Name: proportion, dtype: float64


In [6]:
def calcular_entropia(y_sub):
    """Calcula a entropia de Shannon: H(S) = -sum(p * log2(p))"""
    if len(y_sub) == 0:
        return 0.0
    p = np.bincount(y_sub) / len(y_sub)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def calcular_ganho_informacao(X_sub, y_sub, atributo, threshold):
    """Calcula o Ganho de Informação para um atributo contínuo binarizado num threshold."""
    H_pai = calcular_entropia(y_sub)

    left_mask = X_sub[atributo] <= threshold
    right_mask = ~left_mask

    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return 0.0

    n = len(y_sub)
    H_filhos = (len(y_sub[left_mask]) / n) * calcular_entropia(
        y_sub[left_mask]
    ) + (len(y_sub[right_mask]) / n) * calcular_entropia(y_sub[right_mask])

    return H_pai - H_filhos


# Cálculo manual demonstrativo para o primeiro nível (Nível 0)
entropia_raiz = calcular_entropia(y)
print(f"Entropia da Raiz H(S): {entropia_raiz:.4f}")

gains = {}
for feat in features:
    best_thresh = X[feat].median()
    gain = calcular_ganho_informacao(X, y, feat, best_thresh)
    gains[feat] = (gain, best_thresh)
    print(
        f"Atributo: {feat:20s} | Limiar: {best_thresh:6.2f} | Ganho de Informação: {gain:.4f}"
    )

melhor_feat = max(gains, key=lambda k: gains[k][0])
print(
    f"\nMelhor nó raiz selecionado: {melhor_feat} (Ganho: {gains[melhor_feat][0]:.4f})"
)

Entropia da Raiz H(S): 0.0208
Atributo: dias_sem_chuva       | Limiar:  12.00 | Ganho de Informação: 0.0024
Atributo: precipitacao         | Limiar:  14.62 | Ganho de Informação: 0.0020
Atributo: temperatura_max      | Limiar:  32.96 | Ganho de Informação: 0.0000
Atributo: umidade_relativa     | Limiar:  45.26 | Ganho de Informação: 0.0020
Atributo: velocidade_vento     | Limiar:   8.02 | Ganho de Informação: 0.0000

Melhor nó raiz selecionado: dias_sem_chuva (Ganho: 0.0024)


In [7]:
class Node:

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None,
        depth=0,
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.depth = depth

    def is_leaf(self):
        return self.value is not None


class DecisionTreeFromScratch:

    def __init__(self, max_depth=4, min_samples_split=5):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _best_split(self, X, y):
        best_gain = -1
        split_feat, split_thresh = None, None

        for feat in X.columns:
            thresholds = np.percentile(X[feat], [25, 50, 75])
            for thresh in thresholds:
                gain = calcular_ganho_informacao(X, y, feat, thresh)
                if gain > best_gain:
                    best_gain = gain
                    split_feat = feat
                    split_thresh = thresh

        return split_feat, split_thresh, best_gain

    def _build_tree(self, X, y, depth=0):
        n_samples, n_classes = len(y), len(np.unique(y))

        # Critérios de parada
        if (
            depth >= self.max_depth
            or n_classes == 1
            or n_samples < self.min_samples_split
        ):
            leaf_val = np.bincount(y).argmax() if len(y) > 0 else 0
            return Node(value=leaf_val, depth=depth)

        feat, thresh, gain = self._best_split(X, y)

        if gain <= 1e-4:
            leaf_val = np.bincount(y).argmax()
            return Node(value=leaf_val, depth=depth)

        left_mask = X[feat] <= thresh
        right_mask = ~left_mask

        left_child = self._build_tree(
            X[left_mask], y[left_mask], depth=depth + 1
        )
        right_child = self._build_tree(
            X[right_mask], y[right_mask], depth=depth + 1
        )

        return Node(
            feature=feat,
            threshold=thresh,
            left=left_child,
            right=right_child,
            depth=depth,
        )

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_sample(self, node, x):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_sample(node.left, x)
        return self._predict_sample(node.right, x)

    def predict(self, X):
        return np.array(
            [self._predict_sample(self.root, row) for _, row in X.iterrows()]
        )

In [8]:
folds = cross_validation_split(X, y, k=5, seed=42)

acc_scratch_train, acc_scratch_test = [], []
acc_sklearn_train, acc_sklearn_test = [], []

for train_idx, test_idx in folds:
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    # Modelo do Zero
    tree_custom = DecisionTreeFromScratch(max_depth=4)
    tree_custom.fit(X_train, y_train)
    acc_scratch_train.append(
        accuracy_score(y_train, tree_custom.predict(X_train))
    )
    acc_scratch_test.append(accuracy_score(y_test, tree_custom.predict(X_test)))

    # Modelo Scikit-Learn
    tree_sklearn = DecisionTreeClassifier(max_depth=4, random_state=42)
    tree_sklearn.fit(X_train, y_train)
    acc_sklearn_train.append(
        accuracy_score(y_train, tree_sklearn.predict(X_train))
    )
    acc_sklearn_test.append(
        accuracy_score(y_test, tree_sklearn.predict(X_test))
    )

print("=== RESULTADOS DA ETAPA 1 (VALIÇÃO CRUZADA 5-FOLD) ===")
print(
    f"Árvore do Zero     | Treino: {np.mean(acc_scratch_train)*100:.2f}% | Teste (5-fold): {np.mean(acc_scratch_test)*100:.2f}%"
)
print(
    f"Scikit-Learn Tree  | Treino: {np.mean(acc_sklearn_train)*100:.2f}% | Teste (5-fold): {np.mean(acc_sklearn_test)*100:.2f}%"
)

=== RESULTADOS DA ETAPA 1 (VALIÇÃO CRUZADA 5-FOLD) ===
Árvore do Zero     | Treino: 99.88% | Teste (5-fold): 99.80%
Scikit-Learn Tree  | Treino: 100.00% | Teste (5-fold): 99.70%
